In [1]:
import numpy as np
import networkx as nx
from scipy.special import logsumexp
from sklearn.metrics import normalized_mutual_info_score
import torch
import torch.nn.functional as F
from torch.special import digamma, gammaln
import matplotlib.pyplot as plt

# Bernoulli SBM - DP-SGD

In [17]:
import torch
import torch.nn.functional as F
from torch.func import grad, vmap
from torch.special import digamma, gammaln


# ── DP helpers ────────────────────────────────────────────────────────────────

def accumulate_privacy(log_moments, sigma, q, max_lambda=32):
    for lam in range(1, max_lambda + 1):
        log_moments[lam] += (q ** 2 * lam * (lam + 1)) / ((1.0 - q) * sigma ** 2)
    return log_moments


def get_epsilon(log_moments, delta):
    import math
    return min(
        (log_moments[lam] - math.log(delta)) / lam
        for lam in log_moments
    )


def clip_per_example(grads_list, C):
    """
    grads_list: list of tensors, each shape (L, *param_shape) — per-example
    grads for one parameter, stacked along dim 0. Clips jointly across all
    params in grads_list (one global per-example L2 norm).
    Returns (list of clipped-and-summed tensors shape *param_shape, norms (L,)).
    """
    L = grads_list[0].shape[0]
    flat = torch.cat([g.reshape(L, -1) for g in grads_list], dim=1)
    norms = flat.norm(dim=1)
    scale = (C / (norms + 1e-8)).clamp(max=1.0)

    clipped_sums = []
    for g in grads_list:
        s = scale.view(L, *([1] * (g.dim() - 1)))
        clipped_sums.append((g * s).sum(0))
    return clipped_sums, norms  * scale # norms  * scale  # return scaled norms for debugging


# ── ELBO terms ────────────────────────────────────────────────────────────────

def elbo_L1(A, gamma_logits, log_alpha1, log_alpha2, i_idx, j_idx, temperature):
    gamma      = F.softmax(gamma_logits / temperature, dim=-1)
    alpha1     = F.softplus(log_alpha1)
    alpha2     = F.softplus(log_alpha2)
    alpha_sum  = alpha1 + alpha2
    E_log_b    = digamma(alpha1) - digamma(alpha_sum)
    E_log_1m_b = digamma(alpha2) - digamma(alpha_sum)
    gamma_outer = gamma[i_idx].unsqueeze(2) * gamma[j_idx].unsqueeze(1)
    log_lik     = (A[i_idx, j_idx].view(-1, 1, 1) * E_log_b
                   + (1 - A[i_idx, j_idx]).view(-1, 1, 1) * E_log_1m_b)
    return (gamma_outer * log_lik).sum()


def elbo_rest(gamma_logits, log_alpha1, log_alpha2, log_rho, temperature):
    N, K = gamma_logits.shape
    gamma      = F.softmax(gamma_logits / temperature, dim=-1)
    alpha1     = F.softplus(log_alpha1)
    alpha2     = F.softplus(log_alpha2)
    rho        = F.softplus(log_rho)
    alpha_sum  = alpha1 + alpha2
    E_log_b    = digamma(alpha1) - digamma(alpha_sum)
    E_log_1m_b = digamma(alpha2) - digamma(alpha_sum)
    rho_0      = rho.sum()
    E_log_pi   = digamma(rho) - digamma(rho_0)

    L2 = (gamma * E_log_pi.unsqueeze(0)).sum()

    prior_alpha1 = torch.ones(K, K)
    prior_alpha2 = torch.ones(K, K)
    diag = torch.eye(K, dtype=torch.bool)
    prior_alpha1[diag]  = 3.0
    prior_alpha2[diag]  = 1.0
    prior_alpha1[~diag] = 1.0
    prior_alpha2[~diag] = 3.0
    prior_sum = prior_alpha1 + prior_alpha2
    L4 = ((prior_alpha1 - 1) * E_log_b
          + (prior_alpha2 - 1) * E_log_1m_b
          + gammaln(prior_sum) - gammaln(prior_alpha1) - gammaln(prior_alpha2)).sum()

    L5 = -(gamma * torch.log(gamma + 1e-10)).sum()

    L6 = -(gammaln(rho_0) - gammaln(rho).sum()
           - (rho_0 - K) * digamma(rho_0)
           + ((rho - 1) * digamma(rho)).sum())

    log_B = gammaln(alpha1) + gammaln(alpha2) - gammaln(alpha_sum)
    L7 = (log_B - (alpha1 - 1) * E_log_b - (alpha2 - 1) * E_log_1m_b).sum()

    return L2 + L4 + L5 + L6 + L7


def elbo_sampled(A, gamma_logits, log_alpha1, log_alpha2, log_rho, sample_pct=0.1, temperature=1.0):
    N = gamma_logits.shape[0]
    total_pairs = N * (N - 1)
    n_samples   = max(1, int(sample_pct * total_pairs))
    idx         = torch.randint(0, N, (n_samples * 2, 2))
    idx         = idx[idx[:, 0] != idx[:, 1]][:n_samples]
    i_idx, j_idx = idx[:, 0], idx[:, 1]
    L1   = elbo_L1(A, gamma_logits, log_alpha1, log_alpha2, i_idx, j_idx, temperature)
    rest = elbo_rest(gamma_logits, log_alpha1, log_alpha2, log_rho, temperature)
    return (L1 + rest) / n_samples


# ── per-example loss (functional forms, needed for grad/vmap) ────────────────
# Both loops differentiate the SAME underlying quantity, just w.r.t. different
# leading params. Each takes (leading params..., fixed params..., i, j, L, temperature)
# where "leading params" are whatever is passed to argnums, in a fixed order:
# (log_alpha1, log_alpha2, log_rho, gamma_logits).

def _loss_single(log_alpha1, log_alpha2, log_rho, gamma_logits, A, i, j, L, temperature):
    l1   = elbo_L1(A, gamma_logits, log_alpha1, log_alpha2,
                   i.unsqueeze(0), j.unsqueeze(0), temperature)
    rest = elbo_rest(gamma_logits, log_alpha1, log_alpha2, log_rho, temperature)
    return -(l1 + rest / L)


def build_per_example_grad_fn(param_order, diff_params):
    """
    param_order: full ordered list of param names this loss depends on,
                 e.g. ['log_alpha1', 'log_alpha2', 'log_rho', 'gamma_logits'].
    diff_params: subset of param_order to differentiate w.r.t. (order matters
                 for the returned tuple, but not relative to param_order).
    Builds a vmapped grad function with a symmetric signature: every entry in
    param_order gets an in_dims value (None if not batched), and argnums is
    computed automatically from where diff_params sit in param_order.
    """
    argnums = tuple(param_order.index(p) for p in diff_params)
    in_dims = tuple(None for _ in param_order) + (None, 0, 0, None, None)
    return vmap(grad(_loss_single, argnums=argnums), in_dims=in_dims)


PARAM_ORDER = ['log_alpha1', 'log_alpha2', 'log_rho', 'gamma_logits']


# ── Training ──────────────────────────────────────────────────────────────────

def binary_sbm_estimate_fully_variational(
    A, num_blocks, iter, lr_gamma, lr_beta, schedule_gamma=0.1, sample_pct=0.9,
    T_start=5.0, T_end=0.5,
    gamma_steps=5, beta_steps=5,
    sigma=1.0, C=4.0, target_delta=1e-5,
    sigma_gamma=None, C_gamma=None,   # None → reuse sigma/C for the gamma loop
    schedule_privacy=0.1, schedule_privacy_gamma=0.1
):
    N = A.shape[0]
    K = num_blocks
    sigma_gamma = sigma if sigma_gamma is None else sigma_gamma
    C_gamma     = C if C_gamma is None else C_gamma

    # stash initial values for the C/sigma schedule
    C_init = C
    C_gamma_init = C_gamma

    gamma_logits = torch.randn(N, K).requires_grad_(True)

    log_alpha1_init = torch.full((K, K), 3.0)
    log_alpha2_init = torch.full((K, K), 10.0)
    diag = torch.eye(K, dtype=torch.bool)
    log_alpha1_init[diag] = 10.0
    log_alpha2_init[diag] = 3.0
    log_alpha1 = log_alpha1_init.clone().requires_grad_(True)
    log_alpha2 = log_alpha2_init.clone().requires_grad_(True)
    log_rho    = (torch.randn(K) + 3.0).requires_grad_(True)

    # beta-loop: alpha1, alpha2 clipped+noised jointly; rho raw
    dp_params_beta = [log_alpha1, log_alpha2, log_rho]
    # gamma-loop: gamma_logits clipped+noised on its own
    dp_params_gamma = [gamma_logits]

    # both built the same way: pick which names to differentiate, in the same
    # fixed PARAM_ORDER, so the two builder calls are symmetric
    per_example_grad_fn_beta  = build_per_example_grad_fn(
        PARAM_ORDER, ['log_alpha1', 'log_alpha2', 'log_rho'])
    per_example_grad_fn_gamma = build_per_example_grad_fn(
        PARAM_ORDER, ['gamma_logits'])

    optimizer_gamma = torch.optim.Adam([
        {"params": [gamma_logits], "lr": lr_gamma, "initial_lr": lr_gamma},
    ])
    optimizer_beta = torch.optim.Adam([
        {"params": [log_rho],                "lr": lr_beta, "initial_lr": lr_beta},
        {"params": [log_alpha1, log_alpha2], "lr": lr_beta, "initial_lr": lr_beta},
    ])

    total_pairs = N * (N - 1)
    n_samples   = max(1, int(sample_pct * total_pairs))
    q           = n_samples / total_pairs

    log_moments_beta  = {lam: 0.0 for lam in range(1, 33)}
    log_moments_gamma = {lam: 0.0 for lam in range(1, 33)}

    # fixed call args, in PARAM_ORDER, regardless of which subset is differentiated
    def call_args(i_idx, j_idx, L, temperature):
        return (log_alpha1, log_alpha2, log_rho, gamma_logits, A, i_idx, j_idx, float(L), temperature)

    gamma_grad_history = []
    beta_grad_history = []

    for step in range(iter):
        for pg in optimizer_gamma.param_groups + optimizer_beta.param_groups:
            pg["outer_lr"] = pg["initial_lr"] / (1.0 + schedule_gamma * step)


        # C/sigma OUTER schedule: shrink C, grow sigma, keep C*sigma constant
        f_outer = 1.0 + schedule_privacy * step
        f_outer_gamma = 1.0 + schedule_privacy_gamma * step

        C_outer           = C_init           / f_outer
        C_gamma_outer     = C_gamma_init     / f_outer_gamma

        # ── gamma steps: DP-SGD, per-example clip + noise ────────────────────
        avg_grad_norm_gamma = None
        for gamma_step in range(gamma_steps):
            for pg in optimizer_gamma.param_groups:
                pg["lr"] = pg["outer_lr"] / (1.0 + schedule_gamma * gamma_step)

            temperature = max(T_end, T_start - (T_start - T_end) * (gamma_step / gamma_steps))

            # C/sigma INNER schedule for gamma
            f_inner_gamma = temperature / T_end # T_start/T_end at start, 1 at end
            C_gamma     = C_gamma_outer     / f_inner_gamma

            idx = torch.randint(0, N, (n_samples * 2, 2))
            idx = idx[idx[:, 0] != idx[:, 1]][:n_samples]
            i_idx, j_idx = idx[:, 0], idx[:, 1]
            L = len(i_idx)

            (g_gamma,) = per_example_grad_fn_gamma(*call_args(i_idx, j_idx, L, temperature))

            clipped_sums, norms = clip_per_example([g_gamma], C_gamma)
            gamma_grad = (clipped_sums[0] + torch.randn_like(clipped_sums[0]) * sigma_gamma * 2 * C_gamma) / L
            
            #
            gamma_grad_history.append((clipped_sums[0].detach().clone(), norms.mean().item(), C_gamma))
            #

            avg_grad_norm_gamma = norms.mean().item()

            optimizer_gamma.zero_grad()
            gamma_logits.grad = gamma_grad

            log_moments_gamma = accumulate_privacy(log_moments_gamma, sigma_gamma, q)

            optimizer_gamma.step()

        # ── beta steps: DP-SGD on alpha1, alpha2, rho (all clipped+noised) ──
        avg_grad_norm_beta = None
        for beta_step in range(beta_steps):
            for pg in optimizer_beta.param_groups:
                pg["lr"] = pg["outer_lr"] / (1.0 + schedule_gamma * beta_step)

            # C/sigma INNER schedule for beta
            f_inner_beta = 1.0 + schedule_privacy * beta_step
            C     = C_outer     / f_inner_beta

            idx = torch.randint(0, N, (n_samples * 2, 2))
            idx = idx[idx[:, 0] != idx[:, 1]][:n_samples]
            i_idx, j_idx = idx[:, 0], idx[:, 1]
            L = len(i_idx)

            g_a1, g_a2, g_rho = per_example_grad_fn_beta(*call_args(i_idx, j_idx, L, 1.0))

            clipped_sums, norms = clip_per_example([g_a1, g_a2], C) # Exclude rho from clipping
            dp_grads = [
                (s + torch.randn_like(s) * sigma * 2 * C) / L
                for s in clipped_sums
            ]
            dp_grads.append(g_rho.sum(0) / L) # Append raw rho gradient without clipping or noise
            
            #
            beta_grad_history.append((clipped_sums[0].detach().clone(), norms.mean().item(), C))
            #
            
            
            avg_grad_norm_beta = norms.mean().item()

            optimizer_beta.zero_grad()
            for p, g in zip(dp_params_beta, dp_grads):
                p.grad = g

            log_moments_beta = accumulate_privacy(log_moments_beta, sigma, q)

            optimizer_beta.step()

        if step % 1 == 0:
            with torch.no_grad():
                loss = elbo_sampled(A, gamma_logits, log_alpha1, log_alpha2, log_rho,
                                    sample_pct=sample_pct, temperature=1.0)
            eps_beta  = get_epsilon(log_moments_beta, target_delta)
            eps_gamma = get_epsilon(log_moments_gamma, target_delta)
            print(f"Step {step} | loss: {loss.item():.4f} "
                  f"| grad norm (gamma/beta): {avg_grad_norm_gamma:.4f}/{avg_grad_norm_beta:.4f} "
                  f"| ε_gamma={eps_gamma:.4f} ε_beta={eps_beta:.4f} "
                  f"(simple sum ε={eps_gamma+eps_beta:.4f}) δ={target_delta:.0e}")

    gamma_posterior     = F.softmax(gamma_logits / T_end, dim=-1).detach()
    rho_posterior       = F.softplus(log_rho).detach()
    alpha1_posterior    = F.softplus(log_alpha1).detach()
    alpha2_posterior    = F.softplus(log_alpha2).detach()
    beta_posterior_mean = alpha1_posterior / (alpha1_posterior + alpha2_posterior)

    return gamma_posterior, rho_posterior, beta_posterior_mean, gamma_grad_history, beta_grad_history

In [129]:
seed = 0
# np.random.seed(seed)
# torch.manual_seed(seed)

# Graph construction
# block_sizes = [500,500]
block_sizes = [100, 100]
# block_sizes = [50,50,50,50,50,50]

num_blocks = len(block_sizes)

n = np.sum(block_sizes)
p = 0.2 #1.5*n**(-0.3)
r = 0.02 #0.15*n**(-0.3)
print(f"Intra prob: {p}")
print(f"Inter prob: {r}")

probs = np.eye(num_blocks) * p + (1 - np.eye(num_blocks)) * r

G = nx.stochastic_block_model(block_sizes, probs, seed=seed)
A = nx.to_numpy_array(G)

true_beta = torch.tensor(probs, dtype=torch.float32)  # (K, K)
gamma_posterior, rho_posterior, beta_posterior_mean, gamma_grad, beta_grad = binary_sbm_estimate_fully_variational(
    torch.tensor(A, dtype=torch.float32), num_blocks, iter=5, lr_gamma=3, lr_beta=0.3,
    schedule_gamma=0.1, sample_pct=0.1, T_start=10.0, T_end=1.0, gamma_steps=20, beta_steps=10, 
    sigma=10, C=0.9, 
    sigma_gamma=0.000000000000000000000000000000000000000001,C_gamma=10**5, # -> Effectively no noise on gamma
    target_delta=1e-5,
    schedule_privacy=0.001, schedule_privacy_gamma=0.1
)

block_labels     = np.array([d['block'] for _, d in G.nodes(data=True)])
predicted_labels = np.argmax(gamma_posterior.numpy(), axis=1)
print(f"NMI: {normalized_mutual_info_score(block_labels, predicted_labels)}")
print(f"rho (posterior Dirichlet params): {rho_posterior.numpy()}")
print(f"pi (posterior mean): {(rho_posterior / rho_posterior.sum()).numpy()}")
print(f"\nbeta posterior mean (E[beta_kl]):")
print(np.round(beta_posterior_mean.numpy(), 3))

Intra prob: 0.2
Inter prob: 0.02
Step 0 | loss: -0.6347 | grad norm (gamma/beta): 0.0542/0.1230 | ε_gamma=444444444444444583949379216020877667455071312792246905178183945497844846444987547648.0000 ε_beta=0.3964 (simple sum ε=444444444444444583949379216020877667455071312792246905178183945497844846444987547648.0000) δ=1e-05
Step 1 | loss: -0.4494 | grad norm (gamma/beta): 0.0002/0.0901 | ε_gamma=888888888888888197340678414618722726897599492877789559419168679535080373160006123520.0000 ε_beta=0.4331 (simple sum ε=888888888888888197340678414618722726897599492877789559419168679535080373160006123520.0000) δ=1e-05
Step 2 | loss: -0.4291 | grad norm (gamma/beta): 0.0022/0.0808 | ε_gamma=1333333333333333482248670976556235055695063068180433979163107611087809728298860150784.0000 ε_beta=0.4698 (simple sum ε=1333333333333333482248670976556235055695063068180433979163107611087809728298860150784.0000) δ=1e-05
Step 3 | loss: -0.3793 | grad norm (gamma/beta): 0.0000/0.0805 | ε_gamma=1777777777777779198515

# Edge Flip + Spectral Clustering

In [2]:
import torch
import numpy as np
from sklearn.cluster import SpectralClustering


def edge_flip(A, epsilon):
    """
    Definition 5 from: https://arxiv.org/pdf/2105.12615
    """
    A_perturbed = np.zeros(A.shape)
    for i in range(A.shape[0]):
        for j in range(i+1, A.shape[0]):
            x = np.random.binomial(n=1, p=1/(1+np.exp(epsilon)))
            if x == 1:
                A_perturbed[i,j] = 1 - A[i,j]
                A_perturbed[j,i] = 1 - A[i,j]
            else:
                A_perturbed[i,j] = A[i,j]
                A_perturbed[j,i] = A[i,j]
    return A_perturbed


def spectral_labels(A_np, num_blocks, seed=0):
    """Spectral clustering on an adjacency/affinity matrix."""
    sc = SpectralClustering(
        n_clusters=num_blocks,
        affinity="precomputed",
        assign_labels="kmeans",
        random_state=seed,
    )
    return sc.fit_predict(A_np)


def estimate_beta_dp_edgeflip(A, num_blocks, epsilon, seed=0):
    """
    Estimate block-pair edge probabilities (beta_kl) with epsilon-relationship-DP,
    by applying the symmetric edge-flip mechanism once to the raw adjacency
    matrix, then computing ordinary (non-private) edge proportions on the
    perturbed matrix.

    Block labels are estimated via spectral clustering run on the ALREADY-
    PERTURBED matrix A_flipped, not on the raw A. This matters for privacy:
    clustering on A_flipped is just post-processing of a DP-released quantity
    (free, by the post-processing property of DP), whereas clustering on raw
    A would leak information about the true graph and break the epsilon-DP
    guarantee entirely.

    Args:
        A: (N, N) symmetric 0/1 adjacency matrix (torch tensor or numpy array)
        num_blocks: K
        epsilon: privacy parameter for the edge-flip mechanism
        seed: random_state for spectral clustering's k-means step

    Returns:
        beta_hat: (K, K) symmetric tensor of bias-corrected proportions,
                  clamped to [0, 1]
        labels: (N,) estimated block labels (numpy array), from spectral
                clustering on A_flipped
    """
    is_torch = torch.is_tensor(A)
    A_np = A.numpy() if is_torch else np.asarray(A)

    A_flipped = edge_flip(A_np, epsilon)

    # clustering happens on A_flipped, not A_np -- see docstring
    labels_np = spectral_labels(A_flipped, num_blocks, seed=seed)

    K = num_blocks
    p_flip = 1 / (1 + np.exp(epsilon))

    beta_hat = torch.zeros(K, K)

    for k in range(K):
        mask_k = labels_np == k
        n_k = mask_k.sum()

        for l in range(k, K):
            mask_l = labels_np == l
            n_l = mask_l.sum()

            if k == l:
                sub = A_flipped[np.ix_(mask_k, mask_k)]
                edge_count = sub.sum() / 2.0
                total_pairs = n_k * (n_k - 1) / 2.0
            else:
                sub = A_flipped[np.ix_(mask_k, mask_l)]
                edge_count = sub.sum()
                total_pairs = n_k * n_l

            if total_pairs <= 0:
                beta_hat[k, l] = beta_hat[l, k] = 0.0
                continue

            observed_proportion = edge_count / total_pairs

            denom = 1 - 2 * p_flip
            if abs(denom) < 1e-8:
                corrected = 0.5
            else:
                corrected = (observed_proportion - p_flip) / denom

            corrected = min(1.0, max(0.0, corrected))
            beta_hat[k, l] = corrected
            beta_hat[l, k] = corrected

    return beta_hat, labels_np

In [23]:
seed = 0
# np.random.seed(seed)
# torch.manual_seed(seed)

# Graph construction
block_sizes = [100, 100]

num_blocks = len(block_sizes)

n = np.sum(block_sizes)
p = 0.2
r = 0.02
print(f"Intra prob: {p}")
print(f"Inter prob: {r}")

probs = np.eye(num_blocks) * p + (1 - np.eye(num_blocks)) * r

G = nx.stochastic_block_model(block_sizes, probs, seed=seed)
A = nx.to_numpy_array(G)


beta_dp, estimated_labels = estimate_beta_dp_edgeflip(
    torch.tensor(A, dtype=torch.float32),
    num_blocks=num_blocks,
    epsilon=1.0,
)
print(beta_dp)

Intra prob: 0.2
Inter prob: 0.02
tensor([[0.2312, 0.0021],
        [0.0021, 0.1906]])


# VEM + Gaussian noise for Betas

In [24]:
import torch
import numpy as np
import networkx as nx
from sklearn.metrics import normalized_mutual_info_score


def binary_sbm_estimate(A, num_blocks, iters, tol=1e-16, n_e_steps=1):
    n = A.shape[0]
    K = num_blocks
    eps = 1e-10

    pi = torch.rand(K)
    pi = pi / pi.sum()

    beta = torch.full((K, K), 0.2)
    beta.fill_diagonal_(0.8)
    beta = beta.clamp(eps, 1 - eps)

    gamma = torch.rand(n, K)
    gamma = gamma / gamma.sum(dim=1, keepdim=True)

    prev_elbo = -float('inf')

    for it in range(iters):
        log_beta   = torch.log(beta.clamp(eps, 1 - eps))
        log1m_beta = torch.log((1 - beta).clamp(eps, 1 - eps))

        for _ in range(n_e_steps):
            M1 = gamma @ log_beta.T
            M0 = gamma @ log1m_beta.T
            term2 = A @ M1 + (1 - A) @ M0 - M0
            log_gamma = torch.log(pi + eps).unsqueeze(0) + term2
            log_gamma = log_gamma - torch.logsumexp(log_gamma, dim=1, keepdim=True)
            gamma = torch.exp(log_gamma)

        GtAG   = gamma.T @ A @ gamma
        Gt1mAG = gamma.T @ (1 - A) @ gamma
        diag_correction = torch.sum(gamma * (gamma @ log1m_beta), dim=1).sum()
        data_term    = (GtAG * log_beta).sum() + (Gt1mAG * log1m_beta).sum() - diag_correction
        prior_term   = (gamma * torch.log(pi + eps)).sum()
        entropy_term = -(gamma * torch.log(gamma + eps)).sum()
        elbo = (data_term + prior_term + entropy_term).item()

        pi = gamma.mean(dim=0)
        numerator   = gamma.T @ A @ gamma
        colsum      = gamma.sum(dim=0)
        denominator = torch.outer(colsum, colsum) - gamma.T @ gamma
        beta = (numerator / denominator.clamp_min(eps)).clamp(eps, 1 - eps)
        beta = 0.5 * (beta + beta.T)

        if abs(elbo - prev_elbo) < tol * abs(prev_elbo if prev_elbo != -float('inf') else 1.0):
            break
        prev_elbo = elbo

    return gamma, pi, beta


# ── graph construction ───────────────────────────────────────────────────────
block_sizes = [100, 100]
num_blocks = len(block_sizes)
p, r = 0.2, 0.02
probs = np.eye(num_blocks) * p + (1 - np.eye(num_blocks)) * r
G = nx.stochastic_block_model(block_sizes, probs, seed=0)
A = torch.tensor(nx.to_numpy_array(G), dtype=torch.float32)
block_labels = np.array([d['block'] for _, d in G.nodes(data=True)])

# ── measure success rate over many independent random-init runs ────────────
N = 100
successes = 0
for run in range(N):
    # torch.manual_seed(run)
    gamma, pi, beta = binary_sbm_estimate(A, num_blocks, 100, n_e_steps=1)
    predicted_labels = gamma.argmax(dim=1).numpy()
    nmi = normalized_mutual_info_score(block_labels, predicted_labels)
    successes += (nmi > 0.9)

print(f"n_e_steps=1, {N} independent random-init runs: "
      f"{successes}/{N} = {successes/N*100:.1f}% success rate")

n_e_steps=1, 100 independent random-init runs: 23/100 = 23.0% success rate
